# PBNN — Baseline Exploration (MNIST testbed + tree bit-op baseline)

Colab notebook combining:
1. A full-precision MNIST testbed (drop-in point for a binary model later)
2. A decision-tree baseline on binarized MNIST, with bit-operation counting
   for comparison against a BNN's XNOR+popcount cost

Run cells top to bottom. GPU not required — everything here runs fine on
Colab's default CPU runtime, but you can switch to a GPU runtime
(Runtime > Change runtime type) if you want to try larger models later.

## Setup

In [1]:
!pip install -q torch torchvision scikit-learn


In [2]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.tree import DecisionTreeClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")


device: cpu


## Part 1 — MNIST testbed (full-precision baseline)

Minimal MLP trained on MNIST. This is the scaffold to drop a binary
(weights + activations) model into once a baseline architecture is picked.

In [3]:
class MLP(nn.Module):
    """Plain full-precision MLP — placeholder for a binary model."""

    def __init__(self, hidden=256):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


def get_loaders(batch_size=128, data_dir="./data"):
    tfm = transforms.Compose([transforms.ToTensor()])
    train_ds = datasets.MNIST(data_dir, train=True, download=True, transform=tfm)
    test_ds = datasets.MNIST(data_dir, train=False, download=True, transform=tfm)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)
    return train_loader, test_loader


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total


def train(model, train_loader, test_loader, device, epochs=3, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        t0 = time.time()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            opt.step()
            total_loss += loss.item() * x.size(0)
        train_loss = total_loss / len(train_loader.dataset)
        test_acc = evaluate(model, test_loader, device)
        print(
            f"epoch {epoch+1}/{epochs}  loss={train_loss:.4f}  "
            f"test_acc={test_acc:.4f}  ({time.time()-t0:.1f}s)"
        )
    return model


In [4]:
train_loader, test_loader = get_loaders(batch_size=128)
model = MLP().to(device)
train(model, train_loader, test_loader, device, epochs=3, lr=1e-3)


100%|██████████| 9.91M/9.91M [00:00<00:00, 12.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 361kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.13MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.04MB/s]


epoch 1/3  loss=0.3207  test_acc=0.9578  (21.5s)
epoch 2/3  loss=0.1184  test_acc=0.9678  (11.0s)
epoch 3/3  loss=0.0754  test_acc=0.9726  (11.4s)


MLP(
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=10, bias=True)
)

## Part 2 — Tree algorithm baseline, bit-op counting

Interpretation: a decision tree trained on **binarized** MNIST pixels
(threshold at 0.5), so it's operating on binary inputs just like a BNN
would. At inference, each internal node visited is one bit comparison, so
total bit ops/sample = the decision-path length for that sample.

We compare this against the bit-op cost (XNOR + popcount) of one forward
pass through the 784→256→256→10 MLP shape above, using the XNOR-Net
accounting convention (Rastegari et al., 2016): 1 XNOR op + ~1 popcount op
per weight-activation pair.

In [5]:
def load_binarized_mnist(data_dir="./data"):
    tfm = transforms.Compose([transforms.ToTensor()])
    train_ds = datasets.MNIST(data_dir, train=True, download=True, transform=tfm)
    test_ds = datasets.MNIST(data_dir, train=False, download=True, transform=tfm)

    def to_arrays(ds):
        x = ds.data.numpy().reshape(len(ds), -1).astype(np.float32) / 255.0
        x_bin = (x > 0.5).astype(np.float32)  # binarize, matching BNN-style input
        y = ds.targets.numpy()
        return x_bin, y

    x_train, y_train = to_arrays(train_ds)
    x_test, y_test = to_arrays(test_ds)
    return x_train, y_train, x_test, y_test


def tree_bit_ops_per_sample(tree_clf, X):
    """Internal-node comparisons (= bit ops, since X is binary) visited
    per sample when predicting with a fitted DecisionTreeClassifier.""
    node_indicator = tree_clf.decision_path(X)
    nodes_per_sample = np.diff(node_indicator.indptr)
    comparisons_per_sample = nodes_per_sample - 1  # exclude the leaf itself
    return comparisons_per_sample


def bnn_bitops_mlp(layer_sizes):
    """Bit-op cost (XNOR + popcount) for one forward pass of a fully-binary
    MLP with the given layer sizes, per XNOR-Net accounting: each
    weight-activation pair costs 1 XNOR; popcount reduction costs the same
    order of magnitude."""
    total_xnor = 0
    total_popcount = 0
    for in_f, out_f in zip(layer_sizes[:-1], layer_sizes[1:]):
        total_xnor += in_f * out_f
        total_popcount += in_f * out_f
    return total_xnor, total_popcount, total_xnor + total_popcount


In [6]:
x_train, y_train, x_test, y_test = load_binarized_mnist()

for depth in [5, 10, 15, None]:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=0)
    clf.fit(x_train, y_train)
    acc = clf.score(x_test, y_test)
    bitops = tree_bit_ops_per_sample(clf, x_test)
    depth_label = depth if depth is not None else "unbounded"
    print(
        f"max_depth={depth_label!s:>10}  test_acc={acc:.4f}  "
        f"actual_tree_depth={clf.get_depth()}  "
        f"avg_bitops/sample={bitops.mean():.1f}  "
        f"max_bitops/sample={bitops.max()}"
    )


max_depth=         5  test_acc=0.6409  actual_tree_depth=5  avg_bitops/sample=5.0  max_bitops/sample=5
max_depth=        10  test_acc=0.8273  actual_tree_depth=10  avg_bitops/sample=10.0  max_bitops/sample=10
max_depth=        15  test_acc=0.8687  actual_tree_depth=15  avg_bitops/sample=14.1  max_bitops/sample=15
max_depth= unbounded  test_acc=0.8650  actual_tree_depth=49  avg_bitops/sample=24.7  max_bitops/sample=49


In [7]:
layer_sizes = [784, 256, 256, 10]  # matches the MLP above
xnor, popcount, total = bnn_bitops_mlp(layer_sizes)
print("BNN forward-pass bit-op cost (XNOR-Net accounting):")
print(f"  layer sizes: {layer_sizes}")
print(f"  XNOR ops: {xnor:,}")
print(f"  popcount ops: {popcount:,}")
print(f"  total bit ops/sample: {total:,}")


BNN forward-pass bit-op cost (XNOR-Net accounting):
  layer sizes: [784, 256, 256, 10]
  XNOR ops: 268,800
  popcount ops: 268,800
  total bit ops/sample: 537,600


## Takeaway

The tree is dramatically cheaper in bit operations (tens of ops/sample vs.
~500K for the BNN MLP) but caps out around 86–87% accuracy, well below the
BNN's reported near-state-of-the-art MNIST accuracy (~98–99%, per Hubara et
al., arXiv:1602.02830). Worth raising directly with Liping: is the tree a
legitimate efficiency baseline despite the accuracy gap, or should the
comparison be held at matched accuracy?